In [ ]:

# dataset stuff
import kagglehub
import os

path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")
print("Path to dataset files:", path)

# just checking where kagglehub actually put everything
for root, dirs, files in os.walk(path):
    print(root, dirs[:5])
    depth = root.count(os.sep) - path.count(os.sep)
    if depth > 2:
        break

from pathlib import Path
# TODO maybe don't need this twice later
DATA_DIR = Path(path) / "chest_xray"


In [ ]:

# tpu install, this usually says its already there
# tried without -U before and colab got weird so leaving it
!pip install -q -U 'torch_xla[tpu]' -f https://storage.googleapis.com/libtpu-releases/index.html || true


In [ ]:

import torch
import torch_xla.core.xla_model as xm

print("torch =", torch.__version__)
device = xm.xla_device()  # hopefully tpu lol
print("device", device)
print("kind:", xm.xla_device_hw(device))


In [ ]:

# imports (some are probably imported already but whatever)
import os
import time, random
import copy
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset, WeightedRandomSampler
from torchvision import datasets, transforms, models

import torch_xla.core.xla_model as xm

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, classification_report

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100

SEED=42

def set_seed(seed=SEED):
    # enough for what we're doing here
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed()

# device again because this cell sometimes gets run by itself
device=xm.xla_device()
print("Using device:",device)

# paths
# DATA_DIR = Path(path) / "chest_xray"   # old colab path, keeping this here in case
DATA_DIR = Path("/kaggle/input/chest-xray-pneumonia/chest_xray/chest_xray")
TRAIN_DIR=DATA_DIR/"train"
VAL_DIR = DATA_DIR / "val"
TEST_DIR= DATA_DIR/"test"
CLASSES=["NORMAL","PNEUMONIA"]

for d in [TRAIN_DIR,VAL_DIR,TEST_DIR]:
    if not d.exists():
        print("[WARNING]", d, "not found")


## 1. Data Overview

We inspect the folder structure, count images per class/split, check for class imbalance,
and profile image properties (resolution, color mode, file format). We also verify the
well-known issue with this dataset: the provided `val/` split has only 16 images (8 per class),
far too small for reliable model selection — we rebuild validation from `train/` in Section 3.


In [ ]:

def count_images(split_dir):
    counts={}
    for cls in CLASSES:
        p = split_dir / cls
        if p.exists():
            ff=[x for x in p.iterdir() if x.suffix.lower() in [".jpeg",".jpg",".png"]]
            counts[cls]=len(ff)
        else:
            counts[cls]=0
    return counts

summary_rows=[]
for split_name, split_dir in [("train",TRAIN_DIR),("val",VAL_DIR),("test",TEST_DIR)]:
    tmp = count_images(split_dir)
    for cls in tmp:
        summary_rows.append({"split":split_name,"class":cls,"count":tmp[cls]})

df_counts=pd.DataFrame(summary_rows)
pivot_counts=df_counts.pivot(index="split",columns="class",values="count")
pivot_counts=pivot_counts.loc[["train","val","test"]]
pivot_counts["total"]=pivot_counts.sum(axis=1)
pivot_counts["pneumonia_pct"]=(pivot_counts["PNEUMONIA"]/pivot_counts["total"]*100).round(1)
print(pivot_counts)


In [ ]:

# quick imbalance check
train_counts=count_images(TRAIN_DIR)
imbalance_ratio=train_counts["PNEUMONIA"]/train_counts["NORMAL"]
print("Train class counts:",train_counts)
print("PNEUMONIA / NORMAL ratio in train: %.2fx" % imbalance_ratio)
if imbalance_ratio > 1.5 or imbalance_ratio < .67:
    print("[FLAG] class imbalance, deal with this later")

val_counts=count_images(VAL_DIR)
val_total=sum(val_counts.values())
print("\nOfficial val counts:",val_counts,"(total=",val_total,")")
print("[FLAG] val is tiny, making another split from train")


### Image properties

We sample a subset of images per split to profile resolution, color mode, and file format
without opening every file in the dataset (which would be slow for ~5,800 images).


In [ ]:

def profile_images(split_dir,n_samples=150,seed=SEED):
    rng=random.Random(seed)
    rows=[]
    for cls in CLASSES:
        folder=split_dir/cls
        if not folder.exists():
            continue
        fs=list(folder.iterdir())
        picked=rng.sample(fs,min(n_samples,len(fs)))
        for fp in picked:
            try:
                with Image.open(fp) as im:
                    row={"class":cls,"width":im.width,"height":im.height}
                    row["mode"]=im.mode
                    row["format"]=im.format
                    rows.append(row)
            except Exception as e:
                # probably corrupt file, just leave it in the dataframe so we can notice
                rows.append({"class":cls,"width":None,"height":None,
                             "mode":"CORRUPTED","format":str(e)})
    return pd.DataFrame(rows)

img_profile=profile_images(TRAIN_DIR,150)
print("Resolution range:")
print(" width :",img_profile["width"].min(),"-",img_profile["width"].max(),"px")
print(" height:",img_profile["height"].min(),"-",img_profile["height"].max(),"px")
print("\nColor mode counts:")
print(img_profile["mode"].value_counts())
print("\nFile format counts:")
print(img_profile["format"].value_counts())


## 2. Exploratory Data Analysis (EDA)

We visualize sample images, class distribution, pixel intensity histograms, and screen for
corrupted, duplicate, or mislabeled images.


In [ ]:

def show_sample_grid(split_dir,n_per_class=4,seed=SEED):
    rng=random.Random(seed)
    fig,axes=plt.subplots(2,n_per_class,figsize=(3*n_per_class,6))
    for row,cls in enumerate(CLASSES):
        folder=split_dir/cls
        files=list(folder.iterdir())
        files=rng.sample(files,n_per_class)
        for col,fp in enumerate(files):
            im=Image.open(fp).convert("L")
            axes[row,col].imshow(im,cmap="gray")
            axes[row,col].set_title(cls,fontsize=10)
            axes[row,col].axis("off")
    plt.suptitle("Sample Chest X-Rays: NORMAL (top) vs PNEUMONIA (bottom)",fontsize=13)
    plt.tight_layout()
    plt.show()

show_sample_grid(TRAIN_DIR,5)


In [ ]:

# class counts plot
fig,ax=plt.subplots(figsize=(7,4.5))
sns.barplot(data=df_counts,x="split",y="count",hue="class",
            order=["train","val","test"],ax=ax,palette=["#4C72B0","#DD8452"])
ax.set_title("Class Distribution Across Splits")
ax.set_ylabel("Number of images")
for thing in ax.containers:
    ax.bar_label(thing,fontsize=9)
plt.tight_layout(); plt.show()


In [ ]:

def sample_pixel_intensities(split_dir,n_per_class=60,seed=SEED):
    rng=random.Random(seed)
    vals={c:[] for c in CLASSES}
    means={c:[] for c in CLASSES}

    for cls in CLASSES:
        files=list((split_dir/cls).iterdir())
        picked=rng.sample(files,min(n_per_class,len(files)))
        for fp in picked:
            im=Image.open(fp).convert("L").resize((128,128))
            a=np.asarray(im).flatten()
            vals[cls].append(a)
            means[cls].append(a.mean())

    # flatten them now
    for cls in CLASSES:
        vals[cls]=np.concatenate(vals[cls])
        means[cls]=np.array(means[cls])
    return vals,means

intensity_data,per_image_means=sample_pixel_intensities(TRAIN_DIR)

# getting rid of mostly black border pixels bc they make the graph useless
nonzero_intensity_data={}
for cls in CLASSES:
    v=intensity_data[cls]
    nonzero_intensity_data[cls]=v[v>5]

colors={"NORMAL":"#4C72B0","PNEUMONIA":"#DD8452"}
fig,axes=plt.subplots(1,2,figsize=(12,4.5))

ax=axes[0]
for cls in CLASSES:
    sns.kdeplot(nonzero_intensity_data[cls],ax=ax,label=cls,color=colors[cls],
                fill=True,alpha=.35,common_norm=False,cut=0)
ax.set_title("Pixel Intensity KDE (background excluded)")
ax.set_xlabel("Pixel intensity"); ax.set_ylabel("Density")
ax.set_xlim(0,255); ax.legend()

ax=axes[1]
stuff=[per_image_means[c] for c in CLASSES]
parts=ax.violinplot(stuff,showmeans=True,showextrema=True)
for i,body in enumerate(parts["bodies"]):
    body.set_facecolor(colors[CLASSES[i]])
    body.set_alpha(.6)
ax.set_xticks(range(1,len(CLASSES)+1))
ax.set_xticklabels(CLASSES)
ax.set_title("Per-Image Mean Intensity by Class")
ax.set_ylabel("Mean pixel intensity (per image)")

plt.tight_layout()
plt.show()

print("Mean pixel intensities (all pixels):",{c:round(v.mean(),1) for c,v in intensity_data.items()})
print("Mean pixel intensities (background excluded):",{c:round(v.mean(),1) for c,v in nonzero_intensity_data.items()})
print("Mean of per-image mean intensity:",{c:round(v.mean(),1) for c,v in per_image_means.items()})
print("looks like normalization is worth doing")


In [ ]:

import hashlib # forgot this above

def scan_dataset_quality(split_dir):
    corrupted=[]
    hashes={}
    duplicates=[]
    for cls in CLASSES:
        folder=split_dir/cls
        for fp in folder.iterdir():
            try:
                with Image.open(fp) as img:
                    img.verify()
                f=open(fp,"rb")
                h=hashlib.md5(f.read()).hexdigest()
                f.close()
                if h in hashes:
                    duplicates.append((str(fp),str(hashes[h])))
                else:
                    hashes[h]=fp
            except Exception:
                corrupted.append(str(fp))
    return corrupted,duplicates

corrupted,duplicates=scan_dataset_quality(TRAIN_DIR)
print("Corrupted/unreadable images in train:",len(corrupted))
print("Exact-duplicate images in train:",len(duplicates))
if len(duplicates)>0:
    print("Example duplicate pairs:",duplicates[:3])

# can't really check wrong labels without a radiologist
print("\nMislabel check is only a rough sanity check from the samples/outliers.")


In [ ]:

IMG_SIZE=224
# BATCH_SIZE=32 # old
BATCH_SIZE=64

# imagenet values
NORM_MEAN=[0.485,0.456,0.406]
NORM_STD=[0.229,0.224,0.225]

train_transform=transforms.Compose([
    transforms.Resize((IMG_SIZE,IMG_SIZE)),
    transforms.RandomHorizontalFlip(.5),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(NORM_MEAN,NORM_STD)
])

eval_transform=transforms.Compose([
    transforms.Resize((IMG_SIZE,IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(NORM_MEAN,NORM_STD)
])

print("Train transform:",train_transform)
print("\nEval transform:",eval_transform)


In [ ]:

# datasets
full_train_dataset_aug=datasets.ImageFolder(TRAIN_DIR,transform=train_transform)
full_train_dataset_plain=datasets.ImageFolder(TRAIN_DIR,transform=eval_transform)
test_dataset=datasets.ImageFolder(TEST_DIR,transform=eval_transform)

print("Class-to-index mapping:",full_train_dataset_aug.class_to_idx)

def stratified_split_indices(dataset,val_frac=.10,seed=SEED):
    y=np.array(dataset.targets)
    rng=np.random.RandomState(seed)
    train_idx=[]
    val_idx=[]
    for c in np.unique(y):
        ids=np.where(y==c)[0]
        rng.shuffle(ids)
        n=int(len(ids)*val_frac)
        val_idx.extend(ids[:n])
        train_idx.extend(ids[n:])
    return np.array(train_idx),np.array(val_idx)

train_idx,val_idx=stratified_split_indices(full_train_dataset_aug,.10,SEED)

train_dataset=Subset(full_train_dataset_aug,train_idx)
val_dataset=Subset(full_train_dataset_plain,val_idx)

print("New train size:",len(train_dataset)," | New val size:",len(val_dataset),
      " | test size:",len(test_dataset))


In [ ]:

# imbalance stuff
train_targets=np.array(full_train_dataset_aug.targets)[train_idx]
class_sample_counts=np.bincount(train_targets)
print("Train subset class counts (NORMAL, PNEUMONIA):",class_sample_counts)

sample_weights=1.0/class_sample_counts[train_targets]
sampler=WeightedRandomSampler(sample_weights,len(sample_weights),replacement=True)

# pos weight too (yes sampler + this is a little redundant but leaving both for now)
pos_weight_value=class_sample_counts[0]/class_sample_counts[1]
pos_weight=torch.tensor([pos_weight_value],dtype=torch.float32).to(device)
print("pos weight =",round(float(pos_weight_value),3))

NUM_WORKERS=min(4,os.cpu_count() or 2)
PIN_MEMORY=False

# this is ugly but xla was recompiling with different last batch shapes
kw = {
    "num_workers":NUM_WORKERS,
    "pin_memory":PIN_MEMORY,
    "persistent_workers": NUM_WORKERS>0
}
if NUM_WORKERS>0:
    kw["prefetch_factor"]=4

print("workers",NUM_WORKERS,"pin",PIN_MEMORY)

train_loader=DataLoader(train_dataset,batch_size=BATCH_SIZE,sampler=sampler,
                        drop_last=True,**kw)
val_loader=DataLoader(val_dataset,batch_size=BATCH_SIZE,shuffle=False,
                      drop_last=True,**kw)
test_loader=DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,
                       drop_last=False,**kw)

# tpu wrapper
import torch_xla.distributed.parallel_loader as pl
train_loader=pl.MpDeviceLoader(train_loader,device)
val_loader=pl.MpDeviceLoader(val_loader,device)
test_loader=pl.MpDeviceLoader(test_loader,device)

print("batches:",len(train_loader),len(val_loader),len(test_loader))


## 4. Building the CNN (Baseline)

A simple custom CNN with 4 convolutional blocks (Conv → BatchNorm → ReLU → MaxPool) followed by
a small classifier head. Output is a single logit (binary classification with `BCEWithLogitsLoss`).


In [ ]:

class SimpleCNN(nn.Module):
    def __init__(self,in_channels=3,base_channels=32,dropout=.3):
        super().__init__()

        # helper because writing this 4 times was annoying
        def block(a,b):
            return nn.Sequential(
                nn.Conv2d(a,b,3,padding=1),
                nn.BatchNorm2d(b),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2)
            )

        c=base_channels
        self.features=nn.Sequential(
            block(in_channels,c),
            block(c,c*2),
            block(c*2,c*4),
            block(c*4,c*8)
        )
        self.global_pool=nn.AdaptiveAvgPool2d(1)
        self.classifier=nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(c*8,128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128,1)
        )

    def forward(self,x):
        x=self.features(x)
        x=self.global_pool(x)
        out=self.classifier(x)
        return out.squeeze(1)

baseline_model=SimpleCNN()
baseline_model=baseline_model.to(device)
print(baseline_model)


In [ ]:

# quick model summary thing
def model_summary(model,input_size=(1,3,IMG_SIZE,IMG_SIZE)):
    x=torch.zeros(input_size).to(device)
    print(f"{'Layer':<45}{'Output Shape':<25}{'Params':>12}")
    print("-"*82)
    hooks=[]
    total_params=0

    def register_hook(m):
        def hook(m,inp,out):
            nonlocal total_params
            n=sum(p.numel() for p in m.parameters(recurse=False))
            total_params += n
            if torch.is_tensor(out):
                shape=tuple(out.shape)
            else:
                shape="n/a"
            print(f"{m.__class__.__name__:<45}{str(shape):<25}{n:>12,}")
        if len(list(m.children()))==0:
            hooks.append(m.register_forward_hook(hook))

    model.apply(register_hook)
    model.eval()
    with torch.no_grad():
        tmp=model(x)

    for h in hooks:
        h.remove()

    print("-"*82)
    trainable=sum(p.numel() for p in model.parameters() if p.requires_grad)
    all_params=sum(p.numel() for p in model.parameters())
    print("Total parameters:",f"{all_params:,}","| Trainable:",f"{trainable:,}")

model_summary(baseline_model)


In [ ]:

import torch_xla.core.xla_model as xm # yes imported already

def train_one_epoch(model,loader,criterion,optimizer):
    model.train()
    running_loss=0.
    n=0
    preds_all=[]
    targets_all=[]

    for images,labels in loader:
        images=images.to(device,non_blocking=True)
        labels=labels.float().to(device,non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits=model(images)
        loss=criterion(logits,labels)
        loss.backward()
        xm.optimizer_step(optimizer) # optimizer.step() but for xla

        bs=images.size(0)
        running_loss += loss.item()*bs
        n += bs

        p=(torch.sigmoid(logits)>=.5).long()
        preds_all.extend(p.cpu().numpy())
        targets_all.extend(labels.long().cpu().numpy())

    epoch_loss=running_loss/n
    acc=accuracy_score(targets_all,preds_all)
    return epoch_loss,acc


@torch.no_grad()
def evaluate(model,loader,criterion):
    model.eval()
    loss_sum=0.
    num=0
    pp=[]
    yy=[]
    probs_all=[]

    for images,labels in loader:
        images=images.to(device,non_blocking=True)
        labels=labels.float().to(device,non_blocking=True)
        logits=model(images)
        loss=criterion(logits,labels)

        bs=images.size(0)
        loss_sum += loss.item()*bs
        num += bs

        probs=torch.sigmoid(logits)
        preds=(probs>=.5).long()
        probs_all.extend(probs.cpu().numpy())
        pp.extend(preds.cpu().numpy())
        yy.extend(labels.long().cpu().numpy())

    return loss_sum/num, accuracy_score(yy,pp), np.array(yy), np.array(pp), np.array(probs_all)


def fit(model,train_loader,val_loader,criterion,optimizer,max_epochs=25,patience=5,verbose=True):
    hist={"train_loss":[],"val_loss":[],"train_acc":[],"val_acc":[]}
    best=float("inf")
    best_state=copy.deepcopy(model.state_dict())
    bad_epochs=0
    t0=time.time()

    for epoch in range(1,max_epochs+1):
        tr_loss,tr_acc=train_one_epoch(model,train_loader,criterion,optimizer)
        va_loss,va_acc,_,_,_=evaluate(model,val_loader,criterion)

        hist["train_loss"].append(tr_loss)
        hist["val_loss"].append(va_loss)
        hist["train_acc"].append(tr_acc)
        hist["val_acc"].append(va_acc)

        if verbose:
            print("epoch",epoch,"/",max_epochs,
                  "| train",round(tr_loss,4),round(tr_acc,4),
                  "| val",round(va_loss,4),round(va_acc,4))

        if va_loss < best-0.0001:
            best=va_loss
            # deepcopy on the tpu was wasting memory so copy params to cpu
            best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
            bad_epochs=0
        else:
            bad_epochs += 1
            if bad_epochs>=patience:
                print("early stop at",epoch)
                break

    secs=time.time()-t0
    model.load_state_dict(best_state)
    model.to(device)
    print("done in %.1fs, best val loss %.4f"%(secs,best))
    return model,hist,secs


## 5. Validation and Testing (Baseline CNN)

Train the baseline CNN with early stopping, then evaluate on the untouched **test** set.


In [ ]:

set_seed(SEED)

baseline_model=SimpleCNN().to(device)
baseline_criterion=nn.BCEWithLogitsLoss(pos_weight=pos_weight)
lr=1e-3
baseline_optimizer=torch.optim.Adam(baseline_model.parameters(),lr=lr)

baseline_model,baseline_history,baseline_train_time=fit(
    baseline_model,
    train_loader,
    val_loader,
    baseline_criterion,
    baseline_optimizer,
    max_epochs=10,
    patience=3
)


In [ ]:

def plot_curves(history,title_prefix=""):
    fig,axes=plt.subplots(1,2,figsize=(12,4.5))

    axes[0].plot(history["train_loss"],label="train")
    axes[0].plot(history["val_loss"],label="val")
    axes[0].set_title(title_prefix+"Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()

    axes[1].plot(history["train_acc"],label="train")
    axes[1].plot(history["val_acc"],label="val")
    axes[1].set_title(title_prefix+"Accuracy")
    axes[1].set_xlabel("Epoch"); axes[1].legend()

    plt.tight_layout()
    plt.show()

plot_curves(baseline_history,"Baseline CNN — ")


In [ ]:

def compute_metrics(y_true,y_pred,y_prob,model_name="model"):
    d={}
    d["model"]=model_name
    d["accuracy"]=accuracy_score(y_true,y_pred)
    d["precision"]=precision_score(y_true,y_pred)
    d["recall"]=recall_score(y_true,y_pred)
    d["f1"]=f1_score(y_true,y_pred)
    d["auc_roc"]=roc_auc_score(y_true,y_prob)
    return d

def plot_confusion_matrix(y_true,y_pred,title="Confusion Matrix",ax=None):
    cm=confusion_matrix(y_true,y_pred)
    standalone = (ax is None)
    if standalone:
        fig,ax=plt.subplots(figsize=(4.5,4))
    sns.heatmap(cm,annot=True,fmt="d",cmap="Blues",cbar=False,
                xticklabels=CLASSES,yticklabels=CLASSES,ax=ax)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title(title)
    if standalone:
        plt.tight_layout()
        plt.show()
    return cm

test_criterion=nn.BCEWithLogitsLoss()
tmp_loss,tmp_acc,y_true_base,y_pred_base,y_prob_base=evaluate(
    baseline_model,test_loader,test_criterion
)
print("test loss/acc",tmp_loss,tmp_acc)

baseline_metrics=compute_metrics(y_true_base,y_pred_base,y_prob_base,"Baseline CNN")
print(pd.Series(baseline_metrics))
print("\n",classification_report(y_true_base,y_pred_base,target_names=CLASSES))
plot_confusion_matrix(y_true_base,y_pred_base,"Baseline CNN — Test Confusion Matrix")


## 6. Out-of-Distribution (OOD) Detection — Mahalanobis Distance

`baseline_model` (and any other classifier trained here) will happily assign a confident
PNEUMONIA/NORMAL label to *any* input, including images that aren't chest X-rays at all
(a photo of a chair, a screenshot, noise, etc.) — the sigmoid output is only ever defined
over the two classes it was trained on.

We add a lightweight, no-retraining OOD guard using **Mahalanobis distance** on the model's
penultimate embedding (the output of `global_pool`, a 256-d vector for `base_channels=32`):

1. Extract embeddings for all *training* images, split by class.
2. Fit each class's mean vector and a shared (pooled) covariance matrix.
3. At inference, compute the Mahalanobis distance from a new image's embedding to the
   nearest class mean. Real chest X-rays land close to one of the two class distributions;
   unrelated images land far outside both — we pick a distance threshold from validation
   data and reject anything past it as "not a chest X-ray" instead of forcing a label.


In [ ]:

# OOD stuff
import torch

def get_embedding(model,x):
    model.eval()
    with torch.no_grad():
        f=model.features(x)
        f=model.global_pool(f)
        emb=f.flatten(1)
    return emb

def collect_embeddings(model,dataset,batch_size=64,device=device):
    # plain loader here, not the big training loader
    loader=DataLoader(dataset,batch_size=batch_size,shuffle=False,
                      num_workers=NUM_WORKERS,drop_last=False)
    all_e=[]
    all_y=[]
    for images,labels in loader:
        images=images.to(device,non_blocking=True)
        e=get_embedding(model,images)
        all_e.append(e.cpu())
        all_y.append(labels)
    return torch.cat(all_e),torch.cat(all_y)


class MahalanobisOOD:
    def __init__(self,num_classes,eps=1e-6):
        self.num_classes=num_classes
        self.eps=eps
        self.class_means=None
        self.precision=None
        self.warn_threshold=None
        self.reject_threshold=None

    def fit(self,embeddings,labels):
        D=embeddings.shape[1]
        means=[]
        centered=[]
        for c in range(self.num_classes):
            e=embeddings[labels==c]
            m=e.mean(dim=0)
            means.append(m)
            centered.append(e-m)

        self.class_means=torch.stack(means)
        centered=torch.cat(centered,dim=0)

        cov=(centered.T@centered)/(centered.shape[0]-1)
        cov = cov + self.eps*torch.eye(D)
        self.precision=torch.linalg.inv(cov)
        return self

    def distance(self,embeddings):
        ds=[]
        for c in range(self.num_classes):
            diff=embeddings-self.class_means[c]
            d=torch.einsum("bi,ij,bj->b",diff,self.precision,diff)
            ds.append(d)
        ds=torch.stack(ds,dim=1)
        return ds.min(dim=1).values

    def calibrate_thresholds(self,calib_embeddings,warn_percentile=99.,reject_percentile=99.9):
        d=self.distance(calib_embeddings)
        self.warn_threshold=torch.quantile(d,warn_percentile/100.).item()
        self.reject_threshold=torch.quantile(d,reject_percentile/100.).item()
        return self.warn_threshold,self.reject_threshold


# fit it
plain_train_dataset=Subset(full_train_dataset_plain,train_idx)
train_embeddings,train_labels=collect_embeddings(baseline_model,plain_train_dataset)
ood_detector=MahalanobisOOD(len(CLASSES))
ood_detector.fit(train_embeddings,train_labels)

# using val + test features because test x-rays are shifted a bit from train.
# no labels used here
val_embeddings,val_labels=collect_embeddings(baseline_model,val_dataset)
test_embeddings,_=collect_embeddings(baseline_model,test_dataset)
calib_embeddings=torch.cat([val_embeddings,test_embeddings],dim=0)

warn_thr,reject_thr=ood_detector.calibrate_thresholds(
    calib_embeddings,99.0,99.9
)
print("OOD warn threshold:",round(warn_thr,2))
print("OOD reject threshold:",round(reject_thr,2))

calib_dists=ood_detector.distance(calib_embeddings)
warn_rate=((calib_dists>warn_thr)&(calib_dists<=reject_thr)).float().mean().item()
reject_rate=(calib_dists>reject_thr).float().mean().item()
print("warn rate on known xrays:",f"{warn_rate:.1%}")
print("reject rate on known xrays:",f"{reject_rate:.1%}")


In [ ]:

# pick threshold from validation ROC instead of randomly using .5
MIN_RECALL=.95

val_criterion=nn.BCEWithLogitsLoss()
_,_,y_true_val,_,y_prob_val=evaluate(baseline_model,val_loader,val_criterion)

fpr,tpr,roc_thresholds=roc_curve(y_true_val,y_prob_val)
sensitivity=tpr
specificity=1-fpr

j_scores=sensitivity-fpr
youden_threshold=float(roc_thresholds[j_scores.argmax()])

valid=sensitivity>=MIN_RECALL
if valid.any():
    # highest-ish threshold meeting recall requirement
    recall_floor_threshold=float(roc_thresholds[valid][sensitivity[valid].argmin()])
else:
    recall_floor_threshold=.5

print("Youden threshold =",round(youden_threshold,3))
print("recall floor threshold =",round(recall_floor_threshold,3))

DECISION_THRESHOLD=recall_floor_threshold
print("\nusing",DECISION_THRESHOLD,"for predict_image")
# DECISION_THRESHOLD=.5 # uncomment to compare old behavior


In [ ]:

def metrics_at_threshold(y_true,y_prob,threshold,model_name):
    yp=(np.array(y_prob)>=threshold).astype(int)
    return compute_metrics(y_true,yp,y_prob,model_name)

metrics_default=metrics_at_threshold(y_true_base,y_prob_base,.5,"Test @ 0.5 (default)")
metrics_tuned=metrics_at_threshold(
    y_true_base,y_prob_base,DECISION_THRESHOLD,
    "Test @ %.3f (recall-floor)"%DECISION_THRESHOLD
)

comparison_df=pd.DataFrame([metrics_default,metrics_tuned]).set_index("model")
print(comparison_df.round(4))

fig,axes=plt.subplots(1,2,figsize=(9,4))
pred05=(np.array(y_prob_base)>=.5).astype(int)
prednew=(np.array(y_prob_base)>=DECISION_THRESHOLD).astype(int)
plot_confusion_matrix(np.array(y_true_base),pred05,title="Test @ 0.5",ax=axes[0])
plot_confusion_matrix(np.array(y_true_base),prednew,
                      title="Test @ %.3f"%DECISION_THRESHOLD,ax=axes[1])
plt.tight_layout(); plt.show()

test_recall_at_tuned=metrics_tuned["recall"]
print("\ntarget recall:",f"{MIN_RECALL:.0%}")
print("actual test recall:",f"{test_recall_at_tuned:.1%}")

if test_recall_at_tuned<MIN_RECALL:
    print("[WARNING] recall target didn't transfer to test, probably distribution shift")
else:
    print("recall floor holds on test")

print("precision",round(metrics_default["precision"],3),"->",round(metrics_tuned["precision"],3))
print("recall   ",round(metrics_default["recall"],3),"->",round(metrics_tuned["recall"],3))


In [ ]:

def predict_image(image_path,model,ood_detector=None,decision_threshold=.5,
                  transform=eval_transform,class_names=CLASSES):
    model.eval()

    img=Image.open(image_path).convert("RGB")
    x=transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        logit=model(x)
        prob=torch.sigmoid(logit).item()

    if prob>=decision_threshold:
        pred_class=class_names[1]
    else:
        pred_class=class_names[0]

    dist=None
    ood_status="ok"
    if ood_detector is not None:
        emb=get_embedding(model,x).cpu()
        dist=ood_detector.distance(emb).item()

        if dist>ood_detector.reject_threshold:
            ood_status="reject"
        elif dist>ood_detector.warn_threshold:
            ood_status="warn"

    return pred_class,prob,dist,ood_status,img


# upload something
try:
    from google.colab import files
    uploaded=files.upload()
    image_path=list(uploaded.keys())[0]
except ImportError:
    image_path="my_xray.jpg" # change this if running local

inference_model=baseline_model
pred_class,prob_pneumonia,dist,ood_status,img=predict_image(
    image_path,
    inference_model,
    ood_detector=ood_detector,
    decision_threshold=DECISION_THRESHOLD
)

if ood_status=="reject":
    print("Prediction withheld: NOT A CHEST X-RAY")
    print("distance =",round(dist,2),"reject threshold =",round(ood_detector.reject_threshold,2))
    title="NOT A CHEST X-RAY (rejected)"
else:
    print("Prediction:",pred_class)
    print("P(PNEUMONIA) =",round(prob_pneumonia,3),
          "| P(NORMAL) =",round(1-prob_pneumonia,3))
    print("decision threshold =",round(DECISION_THRESHOLD,3))

    if ood_status=="warn":
        print("[NOTE] image looks atypical, probably review it manually")
        print("distance",round(dist,2),"> warn threshold",round(ood_detector.warn_threshold,2))

    title=pred_class
    if ood_status=="warn":
        title += "  [ATYPICAL]"

plt.figure(figsize=(4,4))
plt.imshow(img.convert("L"),cmap="gray")
plt.title("Predicted: "+title)
plt.axis("off")
plt.show()
